<h1><center>Laboratorio 9: Optimizaci?n de modelos ??</center></h1>

<center><strong>MDS7202: Laboratorio de Programaci?n Cient?fica para Ciencia de Datos - Primavera 2024</strong></center>

### **Cuerpo Docente:**

- Profesores: Ignacio Meza, Sebasti?n Tinoco
- Auxiliar: Eduardo Moya
- Ayudantes: Nicol?s Ojeda, Melanie Pe?a, Valentina Rojas

### Equipo: **SUPER IMPORTANTE - notebooks sin nombre no ser?n revisados**

- Nombre de alumno 1: Joaqu?n De Groote
- Nombre de alumno 2: Vicente Pinochet R.

### **Link de repositorio de GitHub:** [Insertar Enlace](https://github.com/Qajirr/MDS7202-Labs)

### Temas a tratar

- Predicci?n de demanda usando `xgboost`
- B?squeda del modelo ?ptimo de clasificaci?n usando `optuna`
- Uso de pipelines.

### Reglas:

- **Grupos de 2 personas**
- Cualquier duda fuera del horario de clases al foro. Mensajes al equipo docente ser?n respondidos por este medio.
- Prohibidas las copias.
- Pueden usar cualquer matrial del curso que estimen conveniente.
- C?digo que no se pueda ejecutar, no ser? revisado.

### Objetivos principales del laboratorio

- Optimizar modelos usando `optuna`
- Recurrir a t?cnicas de *prunning*
- Forzar el aprendizaje de relaciones entre variables mediante *constraints*
- Fijar un pipeline con un modelo base que luego se ir? optimizando.

El laboratorio deber? ser desarrollado sin el uso indiscriminado de iteradores nativos de python (aka "for", "while"). La idea es que aprendan a exprimir al m?ximo las funciones optimizadas que nos entrega `pandas`, las cuales vale mencionar, son bastante m?s eficientes que los iteradores nativos sobre DataFrames.

# Importamos librerias ?tiles

In [ ]:
!pip install -qq xgboost optuna

# El emprendimiento de Fiu

Tras liderar de manera exitosa la implementaci?n de un proyecto de ciencia de datos para caracterizar los datos generados en Santiago 2023, el misterioso corp?reo **Fiu** se anima y decide levantar su propio negocio de consultor?a en machine learning. Tras varias e intensas negociaciones, Fiu logra encontrar su *primera chamba*: predecir la demanda (cantidad de venta) de una famosa productora de bebidas de calibre mundial. Al ver el gran potencial y talento que usted ha demostrado en el campo de la ciencia de datos, Fiu lo contrata como data scientist para que forme parte de su nuevo emprendimiento.

Para este laboratorio deben trabajar con los datos `sales.csv` subidos a u-cursos, el cual contiene una muestra de ventas de la empresa para diferentes productos en un determinado tiempo.

Para comenzar, cargue el dataset se?alado y visualice a trav?s de un `.head` los atributos que posee el dataset.

<i><p align="center">Fiu siendo felicitado por su excelente desempe?o en el proyecto de caracterizaci?n de datos</p></i>
<p align="center">
  <img src="https://media-front.elmostrador.cl/2023/09/A_UNO_1506411_2440e.jpg">
</p>

In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime

df = pd.read_csv('sales.csv')

df.head()

## 1 Generando un Baseline (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/O-lan6TkadUAAAAC/what-i-wnna-do-after-a-baseline.gif">
</p>

Antes de entrenar un algoritmo, usted recuerda los apuntes de su mag?ster en ciencia de datos y recuerda que debe seguir una serie de *buenas pr?cticas* para entrenar correcta y debidamente su modelo. Despu?s de un par de vueltas, llega a las siguientes tareas:

1. Separe los datos en conjuntos de train (70%), validation (20%) y test (10%). Fije una semilla para controlar la aleatoriedad. [0.5 puntos]
2. Implemente un `FunctionTransformer` para extraer el d?a, mes y a?o de la variable `date`. Guarde estas variables en el formato categorical de pandas. [1 punto]
3. Implemente un `ColumnTransformer` para procesar de manera adecuada los datos num?ricos y categ?ricos. Use `OneHotEncoder` para las variables categ?ricas. `Nota:` Utilice el m?todo `.set_output(transform='pandas')` para obtener un DataFrame como salida del `ColumnTransformer` [1 punto]
4. Guarde los pasos anteriores en un `Pipeline`, dejando como ?ltimo paso el regresor `DummyRegressor` para generar predicciones en base a promedios. [0.5 punto]
5. Entrene el pipeline anterior y reporte la m?trica `mean_absolute_error` sobre los datos de validaci?n. ?C?mo se interpreta esta m?trica para el contexto del negocio? [0.5 puntos]
6. Finalmente, vuelva a entrenar el `Pipeline` pero esta vez usando `XGBRegressor` como modelo **utilizando los par?metros por default**. ?C?mo cambia el MAE al implementar este algoritmo? ?Es mejor o peor que el `DummyRegressor`? [1 punto]
7. Guarde ambos modelos en un archivo .pkl (uno cada uno) [0.5 puntos]

In [ ]:
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import FunctionTransformer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor
import pickle
set_config(transform_output="pandas")

In [ ]:
# Inserte su c?digo ac?
# 1. Separar los datos
train_data, temp_data = train_test_split(df, test_size=0.3, random_state=99)
val_data, test_data = train_test_split(temp_data, test_size=0.33, random_state=99)

In [ ]:
# Definir la funci?n para extraer d?a, mes y a?o
def extract_date_features(df):
    df['date'] = pd.to_datetime(df['date'], dayfirst=True)
    df['day'] = df['date'].dt.day
    df['month'] = df['date'].dt.month
    df['year'] = df['date'].dt.year
    df['day'] = df['day'].astype('category')
    df['month'] = df['month'].astype('category')
    df['year'] = df['year'].astype('category')
    return df.drop(columns='date')

# 2. Crear el transformer
date_transformer = FunctionTransformer(extract_date_features)

In [ ]:
# Seleccionar las columnas categ?ricas y num?ricas
categorical_columns = ['city', 'shop', 'day', 'month', 'year', 'brand',	'container', 'capacity']
numeric_columns = ['id', 'lat',	'long', 'pop', 'price']

# 3. Definir el transformer
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_columns),
        ('cat', OneHotEncoder(sparse_output=False), categorical_columns)
    ]
)
# Configurar salida en formato DataFrame
preprocessor.set_output(transform="pandas")

In [ ]:
# 4. Crear el pipeline con el preprocesador y el DummyRegressor
pipeline_dummy = Pipeline(steps=[
    ('date to categorical', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', DummyRegressor(strategy='mean'))
])

In [ ]:
# 5. Entrenar el modelo
X_train = train_data.drop(columns='quantity')
y_train = train_data['quantity']
pipeline_dummy.fit(X_train, y_train)

X_val = val_data.drop(columns='quantity')
y_val = val_data['quantity']

# Predicciones
y_pred_dummy = pipeline_dummy.predict(X_val)

# Calcular MAE
mae_dummy = mean_absolute_error(y_val, y_pred_dummy)
print(f"MAE del DummyRegressor: {mae_dummy:.2f}")

La m?trica MAE mide la magnitud promedio de los errores en un conjunto de predicciones. En este caso, representa el error promedio en las predicciones de la demanda de ventas de la productora. Un MAE m?s bajo indica un mejor desempe?o.

MAE DummyRegressor= 13735.82
- Aproximadamente el valor promedio de las ventas.

In [ ]:
# 6. Crear el pipeline con XGBRegressor
pipeline_xgb = Pipeline(steps=[
    ('date to categorical', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor())
])

# Entrenar el modelo
pipeline_xgb.fit(X_train, y_train)

# Predicciones
y_pred_xgb = pipeline_xgb.predict(X_val)

# Calcular MAE
mae_xgb = mean_absolute_error(y_val, y_pred_xgb)
print(f"MAE del XGBRegressor: {mae_xgb:.2f}")

MAE XGBRegressor= 2595.22
-  Mejora respecto al Dummy ya que el modelo es capaz de capturar patrones m?s complejos.
-  Es considerablemente un mejor modelo al anterior

In [ ]:
# 7. Guardar el DummyRegressor
with open('dummy_regressor.pkl', 'wb') as f:
    pickle.dump(pipeline_dummy, f)

# Guardar el XGBRegressor
with open('xgb_regressor.pkl', 'wb') as f:
    pickle.dump(pipeline_xgb, f)

## 2. Forzando relaciones entre par?metros con XGBoost (10 puntos)

<p align="center">
  <img src="https://64.media.tumblr.com/14cc45f9610a6ee341a45fd0d68f4dde/20d11b36022bca7b-bf/s640x960/67ab1db12ff73a530f649ac455c000945d99c0d6.gif">
</p>

Un colega aficionado a la econom?a le *sopla* que la demanda guarda una relaci?n inversa con el precio del producto. Motivado para impresionar al querido corp?reo, se propone hacer uso de esta informaci?n para mejorar su modelo realizando las siguientes tareas:

1. Vuelva a entrenar el `Pipeline` con `XGBRegressor`, pero esta vez forzando una relaci?n mon?tona negativa entre el precio y la cantidad. Para aplicar esta restricci?n ap?yese en la siguiente <a href = https://xgboost.readthedocs.io/en/stable/tutorials/monotonic.html>documentaci?n</a>. [6 puntos]

>Hint 1: Para implementar el constraint se le sugiere hacerlo especificando el nombre de la variable. De ser as?, probablemente le sea ?til **mantener el formato de pandas** antes del step de entrenamiento.

>Hint 2: Puede obtener el nombre de las columnas en el paso anterior al modelo regresor mediante el m?todo `.get_feature_names_out()`

2. Luego, vuelva a reportar el `MAE` sobre el conjunto de validaci?n. [1 puntos]

3. ?C?mo cambia el error al incluir esta relaci?n? ?Ten?a raz?n su amigo? [2 puntos]

4. Guarde su modelo en un archivo .pkl [1 punto]

In [ ]:
# Inserte su c?digo ac?
# Obtener nombres de las columnas transformadas
feature_names = pipeline_xgb.named_steps['preprocessor'].get_feature_names_out()
price_index = list(feature_names).index('num__price')

# Configurar las restricciones monot?nicas
relacion_precio = (0,) * len(feature_names)  # Inicializamos relaci?n para cada variable en 0
relacion_precio = list(relacion_precio)
relacion_precio[price_index] = -1  # Forzamos la relaci?n negativa entre precio y cantidad (mon?tona decreciente)
relacion_precio = tuple(relacion_precio)

pipeline_xgb_monotonic = Pipeline(steps=[
    ('date to categorical', date_transformer),
    ('preprocessor', preprocessor),
    ('regressor', XGBRegressor(monotone_constraints=relacion_precio))
])

# Entrenar el modelo
pipeline_xgb_monotonic.fit(X_train, y_train)

In [ ]:
# Predicciones con el nuevo modelo
y_pred_xgb_monotonic = pipeline_xgb_monotonic.predict(X_val)

# Calcular el MAE
mae_xgb_monotonic = mean_absolute_error(y_val, y_pred_xgb_monotonic)
print(f"MAE del XGBRegressor con constraints: {mae_xgb_monotonic:.2f}")


El MAE NO disminuye al incluir la restricci?n, eso indica que la relaci?n inversa entre precio y demanda era falsa y su inclusi?n ha empeorado el rendimiento del modelo.

MAE del XGBRegressor con constraints: 2660.86 > MAE del XGBRegressor: 2595.22

In [ ]:
# Guardar el modelo con restricciones en un archivo .pkl
with open('xgb_regressor_monotonic.pkl', 'wb') as f:
    pickle.dump(pipeline_xgb_monotonic, f)

## 1.3 Optimizaci?n de Hiperpar?metros con Optuna (20 puntos)

<p align="center">
  <img src="https://media.tenor.com/fmNdyGN4z5kAAAAi/hacking-lucy.gif">
</p>

Luego de presentarle sus resultados, Fiu le pregunta si es posible mejorar *aun m?s* su modelo. En particular, le comenta de la optimizaci?n de hiperpar?metros con metodolog?as bayesianas a trav?s del paquete `optuna`. Como usted es un aficionado al entrenamiento de modelos de ML, se propone implementar la descabellada idea de su jefe.

A partir de la mejor configuraci?n obtenida en la secci?n anterior, utilice `optuna` para optimizar sus hiperpar?metros. En particular, se pide que su optimizaci?n considere lo siguiente:

- Fijar una semilla en las instancias necesarias para garantizar la reproducibilidad de resultados
- Utilice `TPESampler` como m?todo de muestreo
- De `XGBRegressor`, optimice los siguientes hiperpar?metros:
    - `learning_rate` buscando valores flotantes en el rango (0.001, 0.1)
    - `n_estimators` buscando valores enteros en el rango (50, 1000)
    - `max_depth` buscando valores enteros en el rango (3, 10)
    - `max_leaves` buscando valores enteros en el rango (0, 100)
    - `min_child_weight` buscando valores enteros en el rango (1, 5)
    - `reg_alpha` buscando valores flotantes en el rango (0, 1)
    - `reg_lambda` buscando valores flotantes en el rango (0, 1)
- De `OneHotEncoder`, optimice el hiperpar?metro `min_frequency` buscando el mejor valor flotante en el rango (0.0, 1.0)

Para ello se pide los siguientes pasos:
1. Implemente una funci?n `objective()` que permita minimizar el `MAE` en el conjunto de validaci?n. Use el m?todo `.set_user_attr()` para almacenar el mejor pipeline entrenado. [10 puntos]
2. Fije el tiempo de entrenamiento a 5 minutos. [1 punto]
3. Optimizar el modelo y reportar el n?mero de *trials*, el `MAE` y los mejores hiperpar?metros encontrados. ?C?mo cambian sus resultados con respecto a la secci?n anterior? ?A qu? se puede deber esto? [3 puntos]
4. Explique cada hiperpar?metro y su rol en el modelo. ?Hacen sentido los rangos de optimizaci?n indicados? [5 puntos]
5. Guardar su modelo en un archivo .pkl [1 punto]

In [ ]:
import optuna
from optuna.samplers import TPESampler
optuna.logging.set_verbosity(optuna.logging.WARNING)
import optuna

from sklearn.metrics import mean_absolute_error
from xgboost import XGBRegressor

# Definir la funci?n de optimizaci?n
def objective(trial):
    # Espacio de b?squeda de hiperpar?metros
    learning_rate = trial.suggest_float("learning_rate", 0.001, 0.1)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0, 1)
    reg_lambda = trial.suggest_float("reg_lambda", 0, 1)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)
    
    # Pipeline con XGBRegressor y optimizaci?n de OneHotEncoder
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_columns),
            ('cat', OneHotEncoder(sparse_output=False, min_frequency=min_frequency), categorical_columns)
        ]
    )
    
    model = XGBRegressor(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=99
    )
    
    pipeline = Pipeline(steps=[
        ('date to categorical', date_transformer),
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Entrenar el pipeline con los datos de entrenamiento
    pipeline.fit(X_train, y_train)
    
    # Predicci?n en el conjunto de validaci?n
    y_pred = pipeline.predict(X_val)
    
    # Calcular el MAE en el conjunto de validaci?n
    mae = mean_absolute_error(y_val, y_pred)
    
    # Almacenar el pipeline entrenado en el trial si es el mejor hasta ahora
    trial.set_user_attr("best_pipeline", pipeline)
    
    return mae

In [ ]:
# Configuraci?n del estudio con Optuna
sampler = TPESampler(seed=99)
study = optuna.create_study(direction="minimize", sampler=sampler)

# Ejecutar la optimizaci?n con un tiempo l?mite de 5 minutos (300 segundos)
study.optimize(objective, timeout=300)

# Mostrar los resultados de la optimizaci?n
best_trial = study.best_trial
print(f"N?mero de trials: {len(study.trials)}")
print(f"Mejor MAE: {best_trial.value}")
print(f"Mejores hiperpar?metros: {best_trial.params}")
# Guardar el mejor pipeline encontrado
best_pipeline = best_trial.user_attrs["best_pipeline"]

# Guardar el modelo en un archivo .pkl
with open('best_xgb_pipeline.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)

learning_rate: controla la tasa de aprendizaje del modelo. Un valor peque?o hace que el modelo aprenda m?s lentamente pero con mayor precisi?n.

n_estimators: el n?mero de ?rboles en el modelo. Un valor mayor permite m?s ?rboles, lo que aumenta la capacidad del modelo, pero tambi?n puede causar sobreajuste.

max_depth: la profundidad m?xima de los ?rboles. Controla la complejidad del modelo. Un valor mayor permite ?rboles m?s profundos pero incrementa el riesgo de sobreajuste.

max_leaves: n?mero m?ximo de hojas por ?rbol. Controla la complejidad de los ?rboles, permitiendo que se formen m?s hojas para capturar m?s patrones.

min_child_weight: el peso m?nimo de una hoja, lo que afecta el n?mero m?nimo de muestras necesarias para dividir un nodo. Valores m?s altos restringen la divisi?n de nodos.

reg_alpha: regularizaci?n L1 (Lasso), que impone una penalizaci?n en las caracter?sticas que no son importantes para forzar la reducci?n de coeficientes.

reg_lambda: regularizaci?n L2 (Ridge), que penaliza grandes coeficientes y ayuda a controlar el sobreajuste.

min_frequency: en el OneHotEncoder, filtra categor?as con frecuencias menores a este valor, lo que puede reducir el ruido causado por categor?as poco frecuentes.

Se utilizan rangos amplios pero estandar que permitan ver resultados variados, tambien pensando en una posible poda mas adelante, y que contaba con recursos computacionales para aprovechar la mayor amplitud, en contextos mas limitados se podrian acotar un poco


## 4. Optimizaci?n de Hiperpar?metros con Optuna y Prunners (17 puntos)

<p align="center">
  <img src="https://i.pinimg.com/originals/90/16/f9/9016f919c2259f3d0e8fe465049638a7.gif">
</p>

Despu?s de optimizar el rendimiento de su modelo varias veces, Fiu le pregunta si no es posible optimizar el entrenamiento del modelo en s? mismo. Despu?s de leer un par de post de personas de dudosa reputaci?n en la *deepweb*, usted llega a la conclusi?n que puede cumplir este objetivo mediante la implementaci?n de **Prunning**.

Vuelva a optimizar los mismos hiperpar?metros que la secci?n pasada, pero esta vez utilizando **Prunning** en la optimizaci?n. En particular, usted debe:

- Responder: ?Qu? es prunning? ?De qu? forma deber?a impactar en el entrenamiento? [2 puntos]
- Redefinir la funci?n `objective()` utilizando `optuna.integration.XGBoostPruningCallback` como m?todo de **Prunning** [10 puntos]
- Fijar nuevamente el tiempo de entrenamiento a 5 minutos [1 punto]
- Reportar el n?mero de *trials*, el `MAE` y los mejores hiperpar?metros encontrados. ?C?mo cambian sus resultados con respecto a la secci?n anterior? ?A qu? se puede deber esto? [3 puntos]
- Guardar su modelo en un archivo .pkl [1 punto]

Nota: Si quieren silenciar los prints obtenidos en el prunning, pueden hacerlo mediante el siguiente comando:

```
optuna.logging.set_verbosity(optuna.logging.WARNING)
```

De implementar la opci?n anterior, pueden especificar `show_progress_bar = True` en el m?todo `optimize` para *m?s sabor*.

Hint: Si quieren especificar par?metros del m?todo .fit() del modelo a trav?s del pipeline, pueden hacerlo por medio de la siguiente sintaxis: `pipeline.fit(stepmodelo__parametro = valor)`

Hint2: Este <a href = https://stackoverflow.com/questions/40329576/sklearn-pass-fit-parameters-to-xgboost-in-pipeline>enlace</a> les puede ser de ayuda en su implementaci?n

In [ ]:
# Inserte su c?digo ac?
import optuna
from optuna.integration import XGBoostPruningCallback
from xgboost import XGBRegressor
from sklearn.metrics import mean_absolute_error
from optuna.samplers import TPESampler

# Redefinir la funci?n objective con pruning
# Definir la funci?n objective con el preprocesamiento adecuado
def objective(trial):
    # Espacio de b?squeda de hiperpar?metros
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.05)
    n_estimators = trial.suggest_int("n_estimators", 50, 1000)
    max_depth = trial.suggest_int("max_depth", 3, 10)
    max_leaves = trial.suggest_int("max_leaves", 0, 100)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 5)
    reg_alpha = trial.suggest_float("reg_alpha", 0, 1)
    reg_lambda = trial.suggest_float("reg_lambda", 0, 1)
    min_frequency = trial.suggest_float("min_frequency", 0.0, 1.0)
    
    # Pipeline con preprocesamiento
    preprocessor = ColumnTransformer(
        transformers=[
            ('num', StandardScaler(), numeric_columns),
            ('cat', OneHotEncoder(sparse_output=False, min_frequency=min_frequency), categorical_columns)
        ]
    )

    # Crear el modelo con XGBRegressor
    model = XGBRegressor(
        learning_rate=learning_rate,
        n_estimators=n_estimators,
        max_depth=max_depth,
        max_leaves=max_leaves,
        min_child_weight=min_child_weight,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        random_state=99,
        use_label_encoder=False,  # No usar LabelEncoder de XGBoost
        enable_categorical=False  # Se asegura que las categ?ricas sean num?ricas ya
    )

    # Definir el pipeline
    pipeline = Pipeline(steps=[
        ('date to categorical', date_transformer),  # Para extraer day, month, year
        ('preprocessor', preprocessor),  # Preprocesar num?ricas y categ?ricas
        ('regressor', model)  # XGBRegressor
    ])

    # Entrenar el pipeline
    pipeline.fit(X_train, y_train)

    # Predicci?n y c?lculo del MAE en el conjunto de validaci?n
    y_pred = pipeline.predict(X_val)
    mae = mean_absolute_error(y_val, y_pred)

    # Almacenar el mejor pipeline si es el mejor hasta ahora
    trial.set_user_attr("best_pipeline", pipeline)

    return mae

In [ ]:
# Fijar el tiempo a 5 minutos y usar pruning
sampler = TPESampler(seed=99)
study = optuna.create_study(direction="minimize", sampler=sampler)

# Silenciar los prints y a?adir barra de progreso
optuna.logging.set_verbosity(optuna.logging.WARNING)

# Ejecutar la optimizaci?n con pruning y un tiempo l?mite de 5 minutos
study.optimize(objective, timeout=300, show_progress_bar=True)

# Mostrar los resultados de la optimizaci?n
best_trial = study.best_trial
print(f"N?mero de trials: {len(study.trials)}")
print(f"Mejor MAE: {best_trial.value}")
print(f"Mejores hiperpar?metros: {best_trial.params}")

In [ ]:
# Guardar el mejor pipeline encontrado
best_pipeline = best_trial.user_attrs["best_pipeline"]

# Guardar el modelo en un archivo .pkl
with open('best_xgb_pipeline_pruning.pkl', 'wb') as f:
    pickle.dump(best_pipeline, f)


La poda es una t?cnica utilizada en la optimizaci?n de hiperpar?metros que permite detener trials que no parecen prometedores antes de que finalicen completamente. En lugar de esperar hasta que un trial termine el proceso de entrenamiento completo, Optuna puede evaluar intermedias metricas de desempe?o y descartar los modelos que no parecen mejorar. Esto ahorra tiempo de c?mputo y recursos, permitiendo que el proceso de optimizaci?n explore m?s configuraciones de hiperpar?metros 

El nuevo MAE es un poco m?s alto que el MAE de la secci?n anterior, lo que significa que el modelo con pruning no ha superado al modelo optimizado sin pruning.
Este peque?o aumento en el MAE podr?a deberse a que el pruning detuvo algunos trials antes de tiempo que, de haber continuado, podr?an haber encontrado una mejor combinaci?n de hiperpar?metros. El pruning es ?til para reducir el tiempo de entrenamiento, pero en algunos casos puede detener trials que m?s adelante habr?an producido mejores resultados.

## 5. Visualizaciones (5 puntos)

<p align="center">
  <img src="https://media.tenor.com/F-LgB1xTebEAAAAd/look-at-this-graph-nickelback.gif">
</p>


Satisfecho con su trabajo, Fiu le pregunta si es posible generar visualizaciones que permitan entender el entrenamiento de su modelo.

A partir del siguiente <a href = https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/005_visualization.html#visualization>enlace</a>, genere las siguientes visualizaciones:

1. Gr?fico de historial de optimizaci?n [1 punto]
2. Gr?fico de coordenadas paralelas [1 punto]
3. Gr?fico de importancia de hiperpar?metros [1 punto]

Comente sus resultados:

4. ?Desde qu? *trial* se empiezan a observar mejoras notables en sus resultados? [0.5 puntos]
5. ?Qu? tendencias puede observar a partir del gr?fico de coordenadas paralelas? [1 punto]
6. ?Cu?les son los hiperpar?metros con mayor importancia para la optimizaci?n de su modelo? [0.5 puntos]

In [ ]:
# Inserte su c?digo ac?
from optuna.visualization import plot_optimization_history

# Graficar el historial de optimizaci?n
optuna_history = plot_optimization_history(study)
optuna_history.show()

In [ ]:
from optuna.visualization import plot_parallel_coordinate

# Graficar coordenadas paralelas
parallel_coordinates = plot_parallel_coordinate(study)
parallel_coordinates.show()

In [ ]:
from optuna.visualization import plot_param_importances

# Graficar la importancia de hiperpar?metros
param_importance = plot_param_importances(study)
param_importance.show()

4. ?Desde qu? trial se empiezan a observar mejoras notables en sus resultados?

- Al observar el gr?fico de historial de optimizaci?n, se puede identificar que desde el trial 7 al 27  los resultados comienzan a mejorar.
5. ?Qu? tendencias puede observar a partir del gr?fico de coordenadas paralelas?

- En el gr?fico de coordenadas paralelas podemos identificar se pueden observar que los hiperpar?metros como learning_rate, max_depth, y n_estimators parecen tener mayor impacto en los resultados del modelo, ya que son los que muestran mayores variaciones en el valor objetivo en funci?n de sus diferentes configuraciones.

6. ?Cu?les son los hiperpar?metros con mayor importancia para la optimizaci?n de su modelo?

- El gr?fico de importancia de hiperpar?metros muestra dos parametros como los principales, min_frequency y n_estimators

## 6. S?ntesis de resultados (3 puntos)

Finalmente:

1. Genere una tabla resumen del MAE en el conjunto de validaci?n obtenido en los 5 modelos entrenados desde Baseline hasta XGBoost con Constraints, Optuna y Prunning. [1 punto]
2. Compare los resultados de la tabla y responda, ?qu? modelo obtiene el mejor rendimiento? [0.5 puntos]
3. Cargue el mejor modelo, prediga sobre el conjunto de **test** y reporte su MAE. [0.5 puntos]
4. ?Existen diferencias con respecto a las m?tricas obtenidas en el conjunto de validaci?n? ?Porqu? puede ocurrir esto? [1 punto]

In [ ]:
# Inserte su c?digo ac?
# Archivos de modelos
modelos_pkl = {
    'Dummy Regressor': 'dummy_regressor.pkl',
    'XGB Regressor Baseline': 'xgb_regressor.pkl',
    'XGB Regressor Monotonic': 'xgb_regressor_monotonic.pkl',
    'XGB Optuna': 'best_xgb_pipeline.pkl',
    'XGB Optuna con Prunning': 'best_xgb_pipeline_pruning.pkl'
}

# Diccionario para almacenar los MAEs
maes = {}

# Cargar y calcular el MAE para cada modelo
for nombre, archivo in modelos_pkl.items():
    with open(archivo, 'rb') as f:
        modelo = pickle.load(f)
    predicciones = modelo.predict(X_val)
    mae = mean_absolute_error(y_val, predicciones)
    maes[nombre] = mae

# Crear tabla resumen de MAEs
df_mae = pd.DataFrame(list(maes.items()), columns=['Modelo', 'MAE Validaci?n'])
print(df_mae)

Basado en el MAE m?s bajo en el conjunto de validaci?n, el mejor modelo es el XGB Optuna con un MAE de 2150.21. Esto indica que, de los cinco modelos entrenados, la versi?n optimizada con Optuna sin pruning tiene la mejor capacidad para predecir los valores en el conjunto de validaci?n.

In [ ]:
# Obtener el mejor modelo con el menor MAE
mejor_modelo_nombre = df_mae.loc[df_mae['MAE Validaci?n'].idxmin(), 'Modelo']
mejor_modelo_archivo = modelos_pkl[mejor_modelo_nombre]

# Cargar el mejor modelo
with open(mejor_modelo_archivo, 'rb') as f:
    mejor_modelo = pickle.load(f)

X_test = test_data.drop(columns='quantity')
y_test = test_data['quantity']

predicciones_test = mejor_modelo.predict(X_test)

# Calcular el MAE en el conjunto de test
mae_test = mean_absolute_error(y_test, predicciones_test)
print(f"MAE en el conjunto de test: {mae_test}")

S?, existe una diferencia entre el MAE del conjunto de validaci?n (2150.21) y el MAE del conjunto de test (2090.28).

Estas diferencias pueden ocurrir por varias razones:

- Suerte en el particionamiento: La variabilidad entre los datos de validaci?n y de test puede ser el resultado de c?mo se dividieron los datos.

- Distribuci?n de los datos: El conjunto de validaci?n y el conjunto de test pueden tener ligeras diferencias en la distribuci?n de las caracter?sticas y los valores objetivo.

# Conclusi?n
Eso ha sido todo para el lab de hoy, recuerden que el laboratorio tiene un plazo de entrega de una semana. Cualquier duda del laboratorio, no duden en contactarnos por mail o U-cursos.

<p align="center">
  <img src="https://media.tenor.com/8CT1AXElF_cAAAAC/gojo-satoru.gif">
</p>

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=87110296-876e-426f-b91d-aaf681223468' target="_blank">
 </img>
Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>